<a href="https://colab.research.google.com/github/Fa-Abdullah/Appie-Responsive-Website/blob/main/Tolerant_Information_Retrieval_System_using_Inverted_Index_and_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Web Scraping

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
base_url = "https://books.toscrape.com/catalogue/page-{}.html"

books_data = []
book_id = 1

num_pages = 50

In [ ]:
for page in range(1, num_pages + 1):

    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    for book in books:
        relative_link = book.h3.a["href"]

        book_url = (
            "https://books.toscrape.com/catalogue/"
            + relative_link.replace("../../../", "")
        )

        book_response = requests.get(book_url)
        book_soup = BeautifulSoup(book_response.text, "html.parser")

        title = book_soup.find("div", class_="product_main").h1.text.strip()

        breadcrumb = book_soup.find("ul", class_="breadcrumb").find_all("li")
        category = breadcrumb[2].text.strip()

        description_tag = book_soup.find("div", id="product_description")

        if description_tag:
            description = description_tag.find_next_sibling("p").text.strip()
        else:
            description = ""

        books_data.append([
            book_id,
            title,
            category,
            description
        ])

        print(f"{book_id} - {title}")

        book_id += 1
        time.sleep(0.2)

1 - A Light in the Attic
2 - Tipping the Velvet
3 - Soumission
4 - Sharp Objects
5 - Sapiens: A Brief History of Humankind
6 - The Requiem Red
7 - The Dirty Little Secrets of Getting Your Dream Job
8 - The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
9 - The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
10 - The Black Maria
11 - Starving Hearts (Triangular Trade Trilogy, #1)
12 - Shakespeare's Sonnets
13 - Set Me Free
14 - Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)
15 - Rip it Up and Start Again
16 - Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991
17 - Olio
18 - Mesaerion: The Best Science Fiction Stories 1800-1849
19 - Libertarianism for Beginners
20 - It's Only the Himalayas
21 - In Her Wake
22 - How Music Works
23 - Foolproof Preserving: A Guide to Small Batch Jams, Jellies, Pickles, Condiments, and More: A Foolproof Guide to Making Small Batch Jams, Je

In [ ]:
df = pd.DataFrame(
    books_data,
    columns=["id", "title", "category", "description"]
)

In [ ]:
print("Done!")
print(f"Saved {len(books_data)} books in books_data.json")

Done!
Saved 1000 books in books_data.json


In [ ]:
df.to_csv("books_data.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Setup NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import string

def clean_description(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)

    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if w not in stop_words]

    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)

df['cleaned_description'] = df['description'].apply(clean_description)

In [ ]:
print(df[['description', 'cleaned_description']].head())

                                         description  \
0  It's hard to imagine a world without A Light i...   
1  "Erotic and absorbing...Written with starling ...   
2  Dans une France assez proche de la nÃ´tre, un ...   
3  WICKED above her hipbone, GIRL across her hear...   
4  From a renowned historian comes a groundbreaki...   

                                 cleaned_description  
0  hard imagine world without light attic nowclas...  
1  erotic absorbingwritten starling powerthe new ...  
2  dans une france assez proche de la ntre un hom...  
3  wicked hipbone girl across heart word like roa...  
4  renowned historian come groundbreaking narrati...  


In [ ]:
df.head()

,id,title,category,description,cleaned_description
0,1,A Light in the Attic,Poetry,It's hard to imagine a world without A Light i...,hard imagine world without light attic nowclas...
1,2,Tipping the Velvet,Historical Fiction,"""Erotic and absorbing...Written with starling ...",erotic absorbingwritten starling powerthe new ...
2,3,Soumission,Fiction,"Dans une France assez proche de la nÃ´tre, un ...",dans une france assez proche de la ntre un hom...
3,4,Sharp Objects,Mystery,"WICKED above her hipbone, GIRL across her hear...",wicked hipbone girl across heart word like roa...
4,5,Sapiens: A Brief History of Humankind,History,From a renowned historian comes a groundbreaki...,renowned historian come groundbreaking narrati...


In [ ]:
df = df.drop(columns=['description'])
df.head()

,id,title,category,cleaned_description
0,1,A Light in the Attic,Poetry,hard imagine world without light attic nowclas...
1,2,Tipping the Velvet,Historical Fiction,erotic absorbingwritten starling powerthe new ...
2,3,Soumission,Fiction,dans une france assez proche de la ntre un hom...
3,4,Sharp Objects,Mystery,wicked hipbone girl across heart word like roa...
4,5,Sapiens: A Brief History of Humankind,History,renowned historian come groundbreaking narrati...


In [ ]:

from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer to extract words
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['cleaned_description'])

# Get the vocabulary (terms)
vocabulary = vectorizer.get_feature_names_out()
print(f"Total unique words in vocabulary: {len(vocabulary)}")


Total unique words in vocabulary: 19837


In [ ]:

# Calculate word frequencies
word_counts = X.toarray().sum(axis=0)
words_freq = pd.DataFrame({'term': vocabulary, 'occurrence': word_counts})
words_freq = words_freq.sort_values(by='occurrence', ascending=False)

# Display the top 20 most common terms
print("Top 20 most frequent terms:")
display(words_freq.head(20))


Top 20 most frequent terms:


,term,occurrence
10259,life,946
11950,new,935
12363,one,900
17811,time,669
19565,world,639
1996,book,616
10572,love,539
19706,year,534
16865,story,521
6589,find,399


## **3-Inverted Index Construction**

In this step, we build an **inverted index** based on the preprocessed text data stored in the `cleaned_description` column.

The goal is to transform the collection of documents into a structured index that maps each term to the documents in which it appears, along with additional statistical information.


In [ ]:
#1: Tokenization from cleaned_description

docs_tokens = []

for text in df['cleaned_description']:
    if pd.isna(text):
        tokens = []
    else:
        tokens = text.split()
    docs_tokens.append(tokens)


# 2: Extract unique terms (vocabulary)

all_terms = []
for tokens in docs_tokens:
    all_terms.extend(tokens)

# remove frequent words to get vocabulary
terms = list(set(all_terms))

print("Number of unique terms:", len(terms))


Number of unique terms: 19856


In [ ]:
#3: Build the Inverted Index

inverted_index = {}

for term in terms:
    term_info = {}

    for i, tokens in enumerate(docs_tokens):
        if term in tokens:
            # Calculate term frequency (TF) in each document
            freq = tokens.count(term)

            term_info[f"Document {i+1}"] = {
                "tf": freq
            }

    # Store document frequency (df) and postings list
    inverted_index[term] = {
        "df": len(term_info),        # number of documents containing the term
        "postings": term_info        # term frequency per document
    }



# 4: Display a sample of the inverted index

for term, data in list(inverted_index.items())[:10]:
    print(term, "->", data)

reflect -> {'df': 8, 'postings': {'Document 91': {'tf': 1}, 'Document 221': {'tf': 1}, 'Document 279': {'tf': 1}, 'Document 437': {'tf': 1}, 'Document 552': {'tf': 1}, 'Document 556': {'tf': 1}, 'Document 822': {'tf': 1}, 'Document 892': {'tf': 1}}}
authoritativelike -> {'df': 1, 'postings': {'Document 299': {'tf': 2}}}
mri -> {'df': 1, 'postings': {'Document 587': {'tf': 1}}}
abusive -> {'df': 2, 'postings': {'Document 730': {'tf': 1}, 'Document 989': {'tf': 2}}}
gangsteron -> {'df': 1, 'postings': {'Document 178': {'tf': 2}}}
salad -> {'df': 14, 'postings': {'Document 58': {'tf': 2}, 'Document 91': {'tf': 1}, 'Document 122': {'tf': 1}, 'Document 200': {'tf': 2}, 'Document 295': {'tf': 2}, 'Document 312': {'tf': 2}, 'Document 335': {'tf': 1}, 'Document 380': {'tf': 1}, 'Document 522': {'tf': 1}, 'Document 530': {'tf': 1}, 'Document 552': {'tf': 1}, 'Document 561': {'tf': 2}, 'Document 591': {'tf': 4}, 'Document 720': {'tf': 1}}}
sexy -> {'df': 18, 'postings': {'Document 76': {'tf': 1}



> The inverted index maps each term to the documents it appears in, along with term frequency and document frequency, enabling efficient search and ranking.


In [ ]:
positional_index = {}

for i, tokens in enumerate(docs_tokens):
    for pos, term in enumerate(tokens):
        if term not in positional_index:
            positional_index[term] = {}

        if f"Document {i+1}" not in positional_index[term]:
            positional_index[term][f"Document {i+1}"] = []

        positional_index[term][f"Document {i+1}"].append(pos)

In [ ]:
for term, data in list(positional_index.items())[:3]:
    print(f"Term: {term}")

    for doc, positions in data.items():
        print(f"   {doc} → positions: {positions}")

    print("-" * 50)

Term: hard
   Document 1 → positions: [0, 37]
   Document 77 → positions: [22]
   Document 123 → positions: [88]
   Document 133 → positions: [7, 44]
   Document 168 → positions: [125]
   Document 186 → positions: [86]
   Document 245 → positions: [100]
   Document 267 → positions: [24, 53]
   Document 288 → positions: [32, 70]
   Document 300 → positions: [17, 54]
   Document 301 → positions: [12, 50]
   Document 325 → positions: [29, 65]
   Document 329 → positions: [82]
   Document 391 → positions: [133]
   Document 395 → positions: [208, 628]
   Document 411 → positions: [98]
   Document 442 → positions: [218]
   Document 473 → positions: [292]
   Document 592 → positions: [13, 50]
   Document 597 → positions: [79]
   Document 602 → positions: [85]
   Document 613 → positions: [138]
   Document 617 → positions: [87]
   Document 638 → positions: [175]
   Document 645 → positions: [147]
   Document 726 → positions: [114]
   Document 730 → positions: [6, 47]
   Document 732 → position

# 4.Search Engine Module

This section implements the search functionality using the constructed positional inverted index.

### Implemented Features:
- Single term search
- Multi-term AND search
- Phrase search
- Result display

##  Single Term Search

- Search for one term  
- Return documents + positions

In [ ]:
def search_term(term, positional_index):
    # Convert term to lowercase
    term = term.lower()

    # Check if term exists in the index
    if term in positional_index:
        results = positional_index[term]

        print(f"Term '{term}' found in:")

        # Iterate over documents and positions
        for doc, positions in results.items():
            print(f"{doc} -> positions: {positions}")
    else:
        print(f"Term '{term}' not found.")

In [ ]:
search_term("love", positional_index)

Term 'love' found in:
Document 1 -> positions: [35, 72]
Document 8 -> positions: [8, 41, 242]
Document 13 -> positions: [68]
Document 20 -> positions: [104]
Document 25 -> positions: [86, 117]
Document 31 -> positions: [32, 70]
Document 32 -> positions: [3, 18, 23, 33, 48, 53, 72, 85, 90, 95]
Document 39 -> positions: [77]
Document 41 -> positions: [25, 62]
Document 42 -> positions: [95]
Document 44 -> positions: [15, 50]
Document 46 -> positions: [37, 77]
Document 47 -> positions: [106]
Document 49 -> positions: [153]
Document 50 -> positions: [103]
Document 58 -> positions: [9, 44, 171]
Document 60 -> positions: [6, 46]
Document 61 -> positions: [84]
Document 73 -> positions: [115]
Document 76 -> positions: [7, 42, 131, 149]
Document 81 -> positions: [89]
Document 85 -> positions: [3, 36]
Document 100 -> positions: [90]
Document 104 -> positions: [29, 63]
Document 107 -> positions: [29, 63, 134, 157]
Document 122 -> positions: [14, 52]
Document 123 -> positions: [95]
Document 125 -> 

##  Multi-Term Search (AND)

- Split query  
- Get common documents  

In [ ]:
def search_multiple_terms(query, positional_index):
    # Split query into terms
    terms = query.lower().split()

    results = None

    # Find intersection of documents containing all terms
    for term in terms:
        if term in positional_index:
            docs = set(positional_index[term].keys())

            if results is None:
                results = docs
            else:
                results = results.intersection(docs)
        else:
            return []

    # Return list of matching documents
    return list(results)

In [ ]:
search_multiple_terms("love story", positional_index)

['Document 999',
 'Document 81',
 'Document 76',
 'Document 47',
 'Document 404',
 'Document 694',
 'Document 100',
 'Document 973',
 'Document 969',
 'Document 237',
 'Document 515',
 'Document 513',
 'Document 708',
 'Document 421',
 'Document 986',
 'Document 329',
 'Document 854',
 'Document 476',
 'Document 242',
 'Document 375',
 'Document 487',
 'Document 662',
 'Document 795',
 'Document 892',
 'Document 595',
 'Document 183',
 'Document 529',
 'Document 137',
 'Document 841',
 'Document 85',
 'Document 164',
 'Document 372',
 'Document 46',
 'Document 728',
 'Document 496',
 'Document 542',
 'Document 734',
 'Document 737',
 'Document 385',
 'Document 481',
 'Document 290',
 'Document 406',
 'Document 503',
 'Document 562',
 'Document 867',
 'Document 874',
 'Document 811',
 'Document 174',
 'Document 327',
 'Document 598',
 'Document 553',
 'Document 634',
 'Document 621',
 'Document 778',
 'Document 433',
 'Document 974',
 'Document 749',
 'Document 263',
 'Document 180',
 '

##  Phrase Search

- Match exact phrase using positions  
- Terms must appear consecutively

In [ ]:
def phrase_search(query, positional_index):
    # Split query into terms
    terms = query.lower().split()

    # Check if all terms exist in the index
    if any(term not in positional_index for term in terms):
        return []

    # Get common documents containing all terms
    common_docs = set(positional_index[terms[0]].keys())

    for term in terms[1:]:
        common_docs = common_docs.intersection(positional_index[term].keys())

    result_docs = []

    # Check positions for exact phrase match
    for doc in common_docs:
        positions_lists = [positional_index[term][doc] for term in terms]

        for pos in positions_lists[0]:
            match = True

            # Check if terms appear consecutively
            for i in range(1, len(terms)):
                if (pos + i) not in positions_lists[i]:
                    match = False
                    break

            if match:
                result_docs.append(doc)
                break

    return result_docs

In [ ]:
phrase_search("love story", positional_index)

['Document 76',
 'Document 708',
 'Document 421',
 'Document 242',
 'Document 375',
 'Document 529',
 'Document 85',
 'Document 874',
 'Document 174',
 'Document 634',
 'Document 263',
 'Document 180',
 'Document 409',
 'Document 212',
 'Document 881',
 'Document 543']

# 5.K-Gram Indexing

  K-Gram Indexing is an important technique used in Information Retrieval systems to enhance search capabilities, especially in:

      Spell correction
      Fuzzy search
      Query suggestion and auto-completion

In [ ]:
def generate_kgrams(word, k=2):
    grams = []
    for i in range(len(word) - k + 1):
        grams.append(word[i:i+k])
    return grams

In [ ]:
def build_kgram_index(terms, k=2):
    kgram_index = {}

    for term in terms:
        grams = generate_kgrams(term, k)

        for gram in grams:
            if gram not in kgram_index:
                kgram_index[gram] = set()

            kgram_index[gram].add(term)

    return kgram_index


k = 2
kgram_index = build_kgram_index(terms, k)

# Jaccard Similarity

  Jaccard Similarity is used to measure the similarity between two sets. It is widely used in text processing to compare words based on their k-grams.

    Formula:
    J(A,B)=∣A∩B∣\∣A∪B∣


	​


In [ ]:
def jaccard_similarity(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))

    if union == 0:
        return 0

    return intersection / union

In [ ]:
#Find Similar Words
def find_similar_words(query, kgram_index, k=2, threshold=0.3):
    query_grams = set(generate_kgrams(query, k))

    candidates = set()

    # Collect candidate words
    for gram in query_grams:
        if gram in kgram_index:
            candidates.update(kgram_index[gram])

    results = []

    for word in candidates:
        word_grams = set(generate_kgrams(word, k))

        score = jaccard_similarity(query_grams, word_grams)

        if score >= threshold:
            results.append((word, score))

    results.sort(key=lambda x: x[1], reverse=True)

    return results

## Edit Distance

- measure of how many insertions, deletions, or substitutions are needed to convert one word into another.

In [ ]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)

    dp = [[0]*(n+1) for _ in range(m+1)]

    for i in range(m+1):
        dp[i][0] = i
    for j in range(n+1):
        dp[0][j] = j

    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i-1][j] + 1,      # delete
                dp[i][j-1] + 1,      # Insertion
                dp[i-1][j-1] + cost  # substitution(replace)
            )

    return dp[m][n]

- best_match with min edit distance

In [ ]:
def best_match(query, candidates):
    best_word = None
    min_dist = float('inf')

    for word, score in candidates:
        dist = edit_distance(query, word)

        if dist < min_dist:
            min_dist = dist
            best_word = word

    return best_word, min_dist

In [ ]:
print(edit_distance("retrival", "retrieval"))

1


## soundex

- algorithm that converts a word into a phonetic code so that words with similar pronunciation have the same representation.

In [ ]:
def soundex(word):
    word = word.upper()

    first_letter = word[0]
# letters with same sound map to same code
    mapping = {
        'BFPV': '1',
        'CGJKQSXZ': '2',
        'DT': '3',
        'L': '4',
        'MN': '5',
        'R': '6'
    }

    def get_code(char):
        for key in mapping:
            if char in key:
                return mapping[key]
        return '0'

    encoded = first_letter

    for char in word[1:]:
        code = get_code(char)
        if code != '0' and code != encoded[-1]:
            encoded += code

    encoded = encoded[:4].ljust(4, '0')

    return encoded

In [ ]:
def correct_query(query, kgram_index):

    candidates = find_similar_words(query, kgram_index)

    if not candidates:
        return "No suggestion found"

    best_word, min_dist = best_match(query, candidates)

    #  Soundex
    query_soundex = soundex(query)
    best_soundex = soundex(best_word)

    return {
        "query": query,
        "candidates": candidates[:5],
        "best_match": best_word,
        "edit_distance": min_dist,
        "soundex_match": query_soundex == best_soundex
    }

In [ ]:
result = correct_query("retrival", kgram_index)

print(result)

{'query': 'retrival', 'candidates': [('retrieval', 0.6666666666666666), ('rival', 0.5714285714285714), ('revival', 0.4444444444444444), ('rivalry', 0.4444444444444444), ('arrival', 0.4444444444444444)], 'best_match': 'retrieval', 'edit_distance': 1, 'soundex_match': True}
